# OpenAI 기반 Agentic Image Generation Hands-on
## From Prompting to Creative Agent

이 Colab 실습은 앞선 **ChatGPT → Copilot → AI Agent → Connected Enterprise AI** 강의 흐름을 이미지 생성 업무에 적용합니다.

핵심 발전 단계:

**Simple Prompt → Structured Prompt → Meta-Prompt → Image Tool → Vision Evaluation → Tool Calling Agent → Human-in-the-Loop**

> 강의 핵심: 좋은 이미지를 직접 프롬프트로만 만드는 것이 아니라, **LLM이 프롬프트를 설계하고 도구를 호출하며 결과를 검증하도록 업무를 위임**합니다.

## 0. 현재 실습에서 사용하는 OpenAI 구성

- **Responses API**: 텍스트, 이미지 입력과 도구 호출을 하나의 흐름에서 오케스트레이션
- **Mainline model**: `gpt-5.5` (교실 환경에서 다른 지원 모델로 변경 가능)
- **Image tool model**: `gpt-image-2.5-flare` (빠른 고품질 생성 실습용)
- 정밀 편집이 중요하면 `gpt-image-2.5-sunburst`로 교체할 수 있습니다.

모델 가용성은 계정/조직 권한에 따라 달라질 수 있으므로 상단의 모델 상수만 바꾸도록 설계했습니다.

In [ ]:
# STEP 0-1. 패키지 설치
!pip -q install -U openai pillow

In [ ]:
# STEP 0-2. OpenAI API Key 설정
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# STEP 0-3. 공통 설정
from openai import OpenAI
from IPython.display import display, Image as IPImage
from pathlib import Path
import base64, json, re, os

client = OpenAI()

TEXT_MODEL = "gpt-5.5"
IMAGE_MODEL = "gpt-image-2.5-flare"
OUTPUT_DIR = Path("/content/openai_agentic_image_lab")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TEXT_MODEL =", TEXT_MODEL)
print("IMAGE_MODEL =", IMAGE_MODEL)
print("OUTPUT_DIR =", OUTPUT_DIR)

# LAB 1. Chatbot 단계 — Simple Prompt

먼저 LLM에게 짧은 사용자 의도를 해석하게 합니다. 아직 이미지 생성은 하지 않습니다.

**관찰 포인트**
- 입력은 짧은 Prompt
- 출력은 Text Response
- Tool 사용 없음
- 아직 Agent가 아니라 **Chatbot형 사용**

In [ ]:
# LAB 1. Simple user intent → text response
simple_intent = "AI Agent와 Connected Enterprise AI를 설명하는 미래형 강의 대표 이미지"

response = client.responses.create(
    model=TEXT_MODEL,
    input=f"다음 사용자 의도를 한 문단의 시각적 장면으로 구체화해 주세요: {simple_intent}",
)

print(response.output_text)

# LAB 2. Image Tool — Basic Text-to-Image

이제 Responses API에 **image_generation built-in tool**을 제공합니다.

구조:

`User Prompt → Mainline Model → Image Generation Tool → PNG Artifact`

이 단계부터 모델은 단순히 답변하는 것이 아니라 **도구를 사용하여 Artifact를 생성**합니다.

In [ ]:
def save_first_generated_image(response, filename: str):
    calls = [x for x in response.output if x.type == "image_generation_call"]
    if not calls:
        raise RuntimeError("No image_generation_call returned.")

    image_call = calls[0]
    path = OUTPUT_DIR / filename
    path.write_bytes(base64.b64decode(image_call.result))
    revised_prompt = getattr(image_call, "revised_prompt", None)
    return path, revised_prompt, response.id

basic_prompt = """
Draw a professional 16:9 hero image for a corporate lecture about AI Agents.
Show an AI orchestration hub connecting presentation, spreadsheet, database, and email workflows.
Modern enterprise visual, clean composition, navy, purple and teal accents, no small readable text.
"""

response = client.responses.create(
    model=TEXT_MODEL,
    input=basic_prompt,
    tools=[{
        "type": "image_generation",
        "model": IMAGE_MODEL,
        "size": "1536x1024",
        "quality": "medium",
        "action": "generate",
    }],
    tool_choice={"type": "image_generation"},
)

basic_path, revised_prompt, basic_response_id = save_first_generated_image(
    response, "lab2_basic.png"
)

print("Saved:", basic_path)
print("Revised prompt:\n", revised_prompt)
display(IPImage(filename=str(basic_path), width=900))

# LAB 3. High-Fidelity Prompt 변수화

업로드된 기존 교안의 핵심 프레임을 OpenAI 실습으로 전환합니다.

**Subject + Action + Lighting + Camera + Style**

영상 Prompt로 확장하기 전, 먼저 한 장면의 품질을 결정하는 변수를 구조화합니다.

In [ ]:
# LAB 3. 구조화 Prompt를 사람이 직접 작성
prompt_parts = {
    "Subject": "A central AI agent orchestration hub in a modern enterprise workspace",
    "Action": "coordinating presentation, spreadsheet, database, email and workflow systems",
    "Lighting": "soft cinematic studio lighting with subtle blue-purple highlights",
    "Camera": "wide-angle 16:9 composition, slightly elevated perspective, clear central hierarchy",
    "Style": "premium corporate technology visualization, photorealistic with restrained UI-like overlays",
    "Constraints": "no tiny readable text, no brand logos, uncluttered composition, presentation-safe negative space",
}

structured_prompt = "\n".join(f"{k}: {v}" for k, v in prompt_parts.items())
print(structured_prompt)

# LAB 4. Meta-Prompting — LLM을 Prompt Engineer로 고용한다

사용자는 세부적인 카메라, 조명, 스타일을 모두 작성하지 않습니다.
LLM이 짧은 사용자 Goal을 **High-Fidelity Image Prompt**로 확장합니다.

이 단계는 앞선 Agent 강의의 개념과 연결됩니다.

`User Intent → Context Interpretation → Prompt Planning → Structured Prompt`

In [ ]:
def build_high_fidelity_prompt(user_intent: str) -> str:
    instruction = f"""
You are a senior visual prompt engineer for enterprise presentation graphics.
Transform the user intent below into ONE production-ready image generation prompt.

USER INTENT:
{user_intent}

Explicitly specify:
- Subject
- Action / relationship between objects
- Environment / scene
- Lighting
- Camera angle and composition
- Visual style
- Color direction
- Negative space for slide title
- Constraints / what to avoid

Return only the final prompt, not explanations.
"""
    r = client.responses.create(model=TEXT_MODEL, input=instruction)
    return r.output_text.strip()

meta_prompt = build_high_fidelity_prompt(
    "AI Agent가 PowerPoint, Excel, 기업 데이터, 이메일을 조율하는 Connected Enterprise AI 강의 표지"
)
print(meta_prompt)

# LAB 5. Meta-Prompt → High-Fidelity Image

이제 LLM이 설계한 Prompt를 image tool에 전달합니다.

수강생은 다음을 비교합니다.
1. LAB 2의 짧은 Prompt 결과
2. LAB 5의 Meta-Prompt 결과
3. 구성, 시선 유도, 스타일 일관성, 강의 슬라이드 활용성

In [ ]:
response = client.responses.create(
    model=TEXT_MODEL,
    input="Draw this image exactly as a presentation-ready hero visual:\n\n" + meta_prompt,
    tools=[{
        "type": "image_generation",
        "model": IMAGE_MODEL,
        "size": "1536x1024",
        "quality": "high",
        "action": "generate",
    }],
    tool_choice={"type": "image_generation"},
)

hf_path, hf_revised_prompt, hf_response_id = save_first_generated_image(
    response, "lab5_high_fidelity.png"
)

print("Saved:", hf_path)
print("Tool revised prompt:\n", hf_revised_prompt)
display(IPImage(filename=str(hf_path), width=900))

# LAB 6. Multi-turn Editing — 생성 후 수정하는 대화형 파이프라인

Responses API의 이미지 생성 Tool은 이전 응답을 이어서 **iterative editing**을 수행할 수 있습니다.

예시 Goal:
- 현재 이미지는 유지
- 제목이 들어갈 왼쪽 상단 여백 확대
- 복잡도 감소
- 강의 슬라이드용 16:9 구성 강화

In [ ]:
follow_up = client.responses.create(
    model=TEXT_MODEL,
    previous_response_id=hf_response_id,
    input=(
        "Edit the previous image. Keep the overall concept and color language, "
        "but reduce visual clutter by about 25%, create cleaner negative space in the upper-left "
        "for a lecture title, and make the enterprise workflow connections easier to read visually."
    ),
    tools=[{
        "type": "image_generation",
        "model": IMAGE_MODEL,
        "size": "1536x1024",
        "quality": "high",
        "action": "edit",
    }],
)

edited_path, edited_revised_prompt, edited_response_id = save_first_generated_image(
    follow_up, "lab6_edited.png"
)

print("Saved:", edited_path)
print("Revised edit prompt:\n", edited_revised_prompt)
display(IPImage(filename=str(edited_path), width=900))

# LAB 7. Vision Evaluation — 생성 결과를 다시 AI가 검토한다

Agent 수준으로 확장하려면 **Generate**만으로는 부족합니다.
생성된 결과가 Goal을 충족했는지 확인하는 **Observe / Evaluate** 단계가 필요합니다.

평가 기준 예:
- Goal alignment
- Composition
- Slide readability
- Professional style
- Visual clutter
- Negative space

In [ ]:
def image_to_data_url(path: Path) -> str:
    mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{encoded}"


def evaluate_image(path: Path, goal: str) -> str:
    rubric = f"""
Evaluate this generated image for the following business goal:
{goal}

Score 0-100 and respond using exactly this compact format:
STATUS: PASS or REVISE
SCORE: integer
STRENGTHS: one sentence
ISSUES: one sentence
REVISION: one concrete prompt instruction

PASS only if score >= 85 and it is presentation-ready.
"""
    r = client.responses.create(
        model=TEXT_MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": rubric},
                {"type": "input_image", "image_url": image_to_data_url(path)},
            ],
        }],
    )
    return r.output_text.strip()

EVAL_GOAL = (
    "A professional 16:9 lecture hero image explaining an AI Agent coordinating enterprise applications. "
    "It must be clean, corporate, visually clear, and leave title space."
)

evaluation = evaluate_image(edited_path, EVAL_GOAL)
print(evaluation)

# LAB 8. Pipeline → Agent

지금까지는 사람이 각 Cell을 순서대로 실행했습니다.

**Pipeline**
`Prompt → Generate → Evaluate → Edit → Save`

이제 Mainline Model에게 Function Tools를 제공하고, **어떤 Tool을 언제 사용할지 선택하게 합니다.**

Agent 패턴:

`Goal → Plan → Tool Call → Observe Tool Result → Decide Next Action → Human Approval → Artifact`

In [ ]:
# LAB 8-1. Agent가 사용할 Tool 구현

def tool_generate_visual(prompt: str, filename: str) -> dict:
    safe_name = Path(filename).name
    if not safe_name.lower().endswith(".png"):
        safe_name += ".png"

    r = client.responses.create(
        model=TEXT_MODEL,
        input="Draw a presentation-ready image:\n" + prompt,
        tools=[{
            "type": "image_generation",
            "model": IMAGE_MODEL,
            "size": "1536x1024",
            "quality": "high",
            "action": "generate",
        }],
        tool_choice={"type": "image_generation"},
    )
    path, revised, _ = save_first_generated_image(r, safe_name)
    return {
        "ok": True,
        "path": str(path),
        "revised_prompt": revised or "",
    }


def parse_eval(text: str) -> dict:
    status = re.search(r"STATUS:\s*(PASS|REVISE)", text, re.I)
    score = re.search(r"SCORE:\s*(\d+)", text, re.I)
    return {
        "status": status.group(1).upper() if status else "REVISE",
        "score": int(score.group(1)) if score else 0,
        "raw": text,
    }


def tool_evaluate_visual(filename: str, goal: str) -> dict:
    path = Path(filename)
    if not path.exists():
        return {"ok": False, "error": f"File not found: {filename}"}
    text = evaluate_image(path, goal)
    result = parse_eval(text)
    result.update({"ok": True, "path": str(path)})
    return result


def tool_request_human_approval(filename: str, rationale: str) -> dict:
    path = Path(filename)
    if path.exists():
        display(IPImage(filename=str(path), width=900))
    print("\nAgent rationale:", rationale)
    answer = input("Approve this image? [y/n]: ").strip().lower()
    return {"approved": answer in {"y", "yes"}, "answer": answer}


def tool_save_manifest(filename: str, summary: str, score: int) -> dict:
    manifest = {
        "final_image": filename,
        "summary": summary,
        "score": score,
        "text_model": TEXT_MODEL,
        "image_model": IMAGE_MODEL,
    }
    path = OUTPUT_DIR / "agent_manifest.json"
    path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"ok": True, "manifest": str(path)}

In [ ]:
# LAB 8-2. Responses API Function Tools 정의
TOOLS = [
    {
        "type": "function",
        "name": "generate_visual",
        "description": "Generate a presentation-ready PNG image from a detailed visual prompt.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {"type": "string"},
                "filename": {"type": "string", "description": "PNG file name only"},
            },
            "required": ["prompt", "filename"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "evaluate_visual",
        "description": "Evaluate a generated image against a business goal and return PASS/REVISE with a score.",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"},
                "goal": {"type": "string"},
            },
            "required": ["filename", "goal"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "request_human_approval",
        "description": "Ask the human to approve a candidate image before finalizing it.",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"},
                "rationale": {"type": "string"},
            },
            "required": ["filename", "rationale"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "save_manifest",
        "description": "Save final artifact metadata after human approval.",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"},
                "summary": {"type": "string"},
                "score": {"type": "integer"},
            },
            "required": ["filename", "summary", "score"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

TOOL_DISPATCH = {
    "generate_visual": tool_generate_visual,
    "evaluate_visual": tool_evaluate_visual,
    "request_human_approval": tool_request_human_approval,
    "save_manifest": tool_save_manifest,
}

## Agent Policy

Agent에게 단순히 “이미지를 만들어”라고 하지 않고, 업무 규칙을 함께 줍니다.

- 먼저 상세 Prompt 설계
- Generate 후 반드시 Evaluate
- `85점 미만`이면 수정된 Prompt로 재생성
- 최대 3회
- PASS 후 Human Approval
- 승인된 결과만 Manifest 저장

이 구조가 **Controlled Autonomy**입니다.

In [ ]:
# LAB 8-3. Agent 실행 루프
AGENT_INSTRUCTIONS = """
You are a Creative Asset Agent for corporate lecture materials.
Your job is to complete the user's visual outcome, not merely describe what to do.

Policy:
1. Convert the user's goal into a detailed, production-ready visual prompt.
2. Call generate_visual to create a candidate PNG.
3. Always call evaluate_visual after each generation.
4. If evaluation is REVISE or score < 85, improve the prompt using evaluator feedback and generate again.
5. Do not exceed 3 generation attempts.
6. When a candidate passes, call request_human_approval.
7. Only after human approval, call save_manifest.
8. If the human rejects the candidate, revise once more if attempts remain.
9. Finish with a concise summary of the final artifact and why it was selected.
"""


def run_creative_agent(user_goal: str, max_rounds: int = 12):
    input_list = [{"role": "user", "content": user_goal}]

    for round_no in range(1, max_rounds + 1):
        print(f"\n===== Agent round {round_no} =====")
        response = client.responses.create(
            model=TEXT_MODEL,
            instructions=AGENT_INSTRUCTIONS,
            tools=TOOLS,
            input=input_list,
        )

        # Keep model outputs as context for the next turn.
        input_list += response.output

        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            print("\nFINAL:\n", response.output_text)
            return response

        for call in tool_calls:
            name = call.name
            args = json.loads(call.arguments)
            print(f"Tool call: {name}({args})")

            if name not in TOOL_DISPATCH:
                result = {"ok": False, "error": f"Unknown tool: {name}"}
            else:
                try:
                    result = TOOL_DISPATCH[name](**args)
                except Exception as e:
                    result = {"ok": False, "error": f"{type(e).__name__}: {e}"}

            print("Tool result:", result)
            input_list.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            })

    raise RuntimeError("Agent exceeded maximum rounds without finishing.")

In [ ]:
# LAB 8-4. 최종 Agent Challenge 실행
USER_GOAL = """
Create the final 16:9 hero visual for a lecture titled
'From Chatbot to Connected Enterprise AI'.

The image should communicate that an AI Agent receives a business goal and coordinates
presentation, spreadsheet, enterprise data, and email/workflow tools to produce an outcome.
It must look premium and professional for a corporate classroom, use navy/purple/teal accents,
avoid tiny readable text and logos, and leave clean negative space for the lecture title.

The final artifact must be suitable for direct use on a PowerPoint title slide.
"""

# 실제 API 비용이 발생하므로 강의자가 준비되었을 때 실행하세요.
# final_response = run_creative_agent(USER_GOAL)

# LAB 9. 결과 확인 및 파일 다운로드

Agent Challenge를 실행했다면 아래 Cell로 결과물을 확인합니다.

In [ ]:
# 생성된 파일 목록 확인
for p in sorted(OUTPUT_DIR.glob("*")):
    print(p.name, p.stat().st_size, "bytes")

In [ ]:
# Colab에서 필요 시 파일 다운로드
# from google.colab import files
# files.download(str(OUTPUT_DIR / "agent_manifest.json"))
# files.download(str(OUTPUT_DIR / "<final-image-name>.png"))

# 실습 정리

### 1. Chatbot
`Prompt → Text`

### 2. Tool-using Model
`Prompt → Image Tool → Artifact`

### 3. Meta-Prompt Pipeline
`Simple Intent → LLM Prompt Engineer → Image Tool`

### 4. Agent
`Goal → Plan → Generate → Evaluate → Revise → Human Approval → Save`

앞선 Agent 강의와 동일한 패턴입니다.

> **Agent = Model + Context + Tools + Execution Loop + Policy + Human Control**

다음 Session에서는 이 구조를 영상 생성으로 확장할 수 있습니다.

**Image Prompt**
`Subject + Action + Lighting + Camera + Style`

→ **Video Prompt**
`Subject + Action + Scene + Camera Movement + Motion + Temporal Sequence + Duration + Audio`

---
## 공식 참고 문서

- OpenAI Image generation tool: https://developers.openai.com/api/docs/guides/tools-image-generation
- OpenAI Image generation guide: https://developers.openai.com/api/docs/guides/image-generation
- OpenAI Function calling: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Images and vision: https://developers.openai.com/api/docs/guides/images-vision

> API와 모델 이름은 변경될 수 있으므로 수업 직전 공식 문서를 확인하세요.